### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import string
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer

### Load dataset

In [2]:
train_data = pd.read_csv('train.txt', sep='\t', header=None, names=['title', 'from', 'genre', 'director', 'plot'])

test_data = pd.read_csv('test_no_labels.txt', sep='\t', header=None, names=['title', 'from', 'director', 'plot'])

### Pre-processing

In [3]:
stop_words = set(stopwords.words('english')) # Maybe reduce the number of stopwords?

def clean_text(text):
    text = text.lower()  # Lowercase all characters
    text = re.sub(r'@\S+', '', text)  # Remove Twitter handles
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'pic.\S+', '', text)
    text = re.sub(r"[^a-zA-Z+']", ' ', text)  # Keep only characters
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text + ' ')  # Keep words with length > 1 only
    text = "".join([i for i in text if i not in string.punctuation])
    words = nltk.word_tokenize(text)
    stopwords = nltk.corpus.stopwords.words('english')  # Remove stopwords
    text = " ".join([i for i in words if i not in stopwords and len(i) > 2])
    text = re.sub("\s[\s]+", " ", text).strip()  # Remove repeated/leading/trailing spaces
    return text

train_data['Text_cleaning'] = train_data['plot'].apply(clean_text)
test_data['Text_cleaning'] = test_data['plot'].apply(clean_text)

### Training the Model

In [4]:
vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(train_data['Text_cleaning'])
X_test = vectorizer.transform(test_data['Text_cleaning'])

# Split the data into training and validation sets
X = X_train
y = train_data['genre']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train a Multinomial Naive Bayes classifier
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

MultinomialNB()

### Model Validation (Train 80% - Test 20%)

In [5]:
# Make predictions on the validation set
y_pred = classifier.predict(X_val)

# Evaluate the performance of the model
accuracy = accuracy_score(y_val, y_pred)
print("Validation Accuracy:", accuracy)
print(classification_report(y_val, y_pred))

Validation Accuracy: 0.6376631448104413
              precision    recall  f1-score   support

      action       0.60      0.62      0.61       202
   animation       0.92      0.68      0.78       115
      comedy       0.56      0.45      0.50       223
       crime       0.81      0.16      0.26       108
       drama       0.50      0.69      0.58       319
      horror       0.70      0.87      0.78       223
     romance       0.55      0.55      0.55       186
      sci-fi       0.75      0.23      0.35        39
     western       0.91      0.93      0.92       194

    accuracy                           0.64      1609
   macro avg       0.70      0.57      0.59      1609
weighted avg       0.66      0.64      0.62      1609



### Model Predictions

In [6]:
# Use the trained model to make predictions on the test data
X_test_predictions = classifier.predict(X_test)

# Save the predictions to 'results.txt' with each predicted genre in a new line
with open('results.txt', 'w') as file:
    for genre in X_test_predictions:
        file.write(genre + '\n')

print("Predictions saved to results.txt")

Predictions saved to results.txt
